In [1]:
import os
from collections import Counter
import pandas as pd
from sklearn.model_selection import train_test_split

DATASET_DIR = "/kaggle/input/datasets/salviohexia/isic-2019-skin-lesion-images-for-classification"

classes = sorted([
    d for d in os.listdir(DATASET_DIR)
    if os.path.isdir(os.path.join(DATASET_DIR, d))
    and not d.startswith(".")
])

print("Classes:", classes)

# Collect image paths and labels
image_paths = []
labels = []

for cls in classes:
    cls_dir = os.path.join(DATASET_DIR, cls)

    for fname in os.listdir(cls_dir):
        if fname.lower().endswith((".jpg", ".jpeg", ".png")):
            image_paths.append(os.path.join(cls_dir, fname))
            labels.append(cls)

df = pd.DataFrame({
    "path": image_paths,
    "label": labels
})

print("\nTotal images:", len(df))

print("\nOriginal Class Counts")
print(df["label"].value_counts().sort_index())

# -------------------------
# 80 / 10 / 10 Stratified
# -------------------------

train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    stratify=df["label"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=42
)

print("\nTrain Size :", len(train_df))
print("Val Size   :", len(val_df))
print("Test Size  :", len(test_df))

print("\nTrain Class Counts")
print(train_df["label"].value_counts().sort_index())

print("\nVal Class Counts")
print(val_df["label"].value_counts().sort_index())

print("\nTest Class Counts")
print(test_df["label"].value_counts().sort_index())

Classes: ['AK', 'BCC', 'BKL', 'DF', 'MEL', 'NV', 'SCC', 'VASC']

Total images: 25331

Original Class Counts
label
AK        867
BCC      3323
BKL      2624
DF        239
MEL      4522
NV      12875
SCC       628
VASC      253
Name: count, dtype: int64

Train Size : 20264
Val Size   : 2533
Test Size  : 2534

Train Class Counts
label
AK        694
BCC      2658
BKL      2099
DF        191
MEL      3618
NV      10300
SCC       502
VASC      202
Name: count, dtype: int64

Val Class Counts
label
AK        87
BCC      332
BKL      262
DF        24
MEL      452
NV      1287
SCC       63
VASC      26
Name: count, dtype: int64

Test Class Counts
label
AK        86
BCC      333
BKL      263
DF        24
MEL      452
NV      1288
SCC       63
VASC      25
Name: count, dtype: int64


In [2]:
import os
import cv2
import numpy as np
from PIL import Image, ImageFile

# Prevent OpenCV multi-threading deadlocks with PyTorch DataLoader worker forks
cv2.setNumThreads(0)
cv2.ocl.setUseOpenCL(False)

# Prevent crashes on truncated/corrupted ISIC JPEG images
ImageFile.LOAD_TRUNCATED_IMAGES = True

import torchvision.transforms as transforms
from torchvision.transforms import InterpolationMode

IMG_SIZE = 224


# =====================================================
# CLAHE
# =====================================================

class CLAHETransform:
    def __init__(self, clip_limit=2.0, tile_grid_size=(8, 8)):
        self.clip_limit = clip_limit
        self.tile_grid_size = tile_grid_size

    def __call__(self, img):
        img = np.array(img)

        # RGB -> LAB
        lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)

        l, a, b = cv2.split(lab)

        clahe = cv2.createCLAHE(
            clipLimit=self.clip_limit,
            tileGridSize=self.tile_grid_size
        )

        l = clahe.apply(l)

        lab = cv2.merge((l, a, b))

        img = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

        return Image.fromarray(img)


# =====================================================
# NORMALIZATION
# =====================================================

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]


# =====================================================
# TRAIN TRANSFORM
# =====================================================

train_transform = transforms.Compose([

    transforms.Resize((256, 256)),

    CLAHETransform(),

    transforms.RandomResizedCrop(
        IMG_SIZE,
        scale=(0.85, 1.0),
        ratio=(0.9, 1.1),
        interpolation=InterpolationMode.BILINEAR
    ),

    transforms.RandomHorizontalFlip(p=0.5),

    transforms.RandomRotation(
        degrees=20,
        interpolation=InterpolationMode.BILINEAR
    ),

    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.10,
        hue=0.02
    ),

    transforms.ToTensor(),

    transforms.RandomErasing(
        p=0.25,
        scale=(0.02, 0.10),
        ratio=(0.3, 3.3)
    ),

    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])


# =====================================================
# VALIDATION / TEST
# =====================================================

val_test_transform = transforms.Compose([

    transforms.Resize((IMG_SIZE, IMG_SIZE)),

    CLAHETransform(),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=IMAGENET_MEAN,
        std=IMAGENET_STD
    )
])


In [3]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image

class ISICDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

        classes = sorted(self.df["label"].unique())
        self.class_to_idx = {
            cls: idx for idx, cls in enumerate(classes)
        }

        # Pre-extract image paths and mapped integer labels to avoid Pandas .iloc locks in multiprocessing workers
        self.image_paths = self.df["path"].tolist()
        self.labels = [self.class_to_idx[l] for l in self.df["label"].tolist()]

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        image = Image.open(path).convert("RGB")
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label


In [4]:
class_names = sorted(train_df["label"].unique())

print("Classes:")
for i, cls in enumerate(class_names):
    print(i, cls)

label_map = {
    cls: idx
    for idx, cls in enumerate(class_names)
}

idx_to_class = {
    idx: cls
    for cls, idx in label_map.items()
}

df["label"] = df["label"].map(label_map)

print(label_map)



Classes:
0 AK
1 BCC
2 BKL
3 DF
4 MEL
5 NV
6 SCC
7 VASC
{'AK': 0, 'BCC': 1, 'BKL': 2, 'DF': 3, 'MEL': 4, 'NV': 5, 'SCC': 6, 'VASC': 7}


In [5]:
import os

train_dataset = ISICDataset(train_df, train_transform)
val_dataset   = ISICDataset(val_df, val_test_transform)
test_dataset  = ISICDataset(test_df, val_test_transform)

BATCH_SIZE = 128

# Optimal worker count for Kaggle/Colab containers (matches 2 vCPUs and avoids /dev/shm shared-memory exhaustion)
NUM_WORKERS = min(2, os.cpu_count() or 1)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True if NUM_WORKERS > 0 else False,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True if NUM_WORKERS > 0 else False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True if NUM_WORKERS > 0 else False
)

print(f"DataLoaders initialized successfully with num_workers={NUM_WORKERS}, batch_size={BATCH_SIZE}")


DataLoaders initialized successfully with num_workers=2, batch_size=128


In [6]:
images, labels = next(iter(train_loader))

print("Images shape:", images.shape)
print("Labels shape:", labels.shape)

print("Classes:")
print(class_names)

print("Number of classes:",
      len(class_names))

Images shape: torch.Size([128, 3, 224, 224])
Labels shape: torch.Size([128])
Classes:
['AK', 'BCC', 'BKL', 'DF', 'MEL', 'NV', 'SCC', 'VASC']
Number of classes: 8


In [7]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import models
from tqdm.auto import tqdm

# =====================================================
# DEVICE
# =====================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

# =====================================================
# EFFECTIVE NUMBER WEIGHTS
# =====================================================

class_counts = (
    train_df["label"]
    .value_counts()
    .sort_index()
    .values
)

print("Class Counts:", class_counts)

beta = 0.999

effective_num = 1.0 - np.power(beta, class_counts)

weights = (1.0 - beta) / effective_num

weights = weights / weights.sum() * len(weights)

weights = torch.tensor(
    weights,
    dtype=torch.float32
).to(device)

print("Class Weights:")
print(weights)



Device: cuda
Class Counts: [  694  2658  2099   191  3618 10300   502   202]
Class Weights:
tensor([0.7996, 0.4304, 0.4561, 2.3011, 0.4113, 0.4003, 1.0138, 2.1874],
       device='cuda:0')


In [8]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import Counter

# ============================================================
# EMA FEATURE QUEUE WITH REAL-TIME DISTRIBUTION & TURNOVER TRACKING
# ============================================================

class EMAFeatureQueue:

    def __init__(
        self,
        feature_dim=512,
        queue_size=4096,
        momentum=0.99,
        num_classes=8,
        device="cpu"
    ):
        self.queue_size = queue_size
        self.momentum = momentum
        self.feature_dim = feature_dim
        self.num_classes = num_classes
        self.device = device

        self.features = torch.zeros(
            queue_size,
            feature_dim,
            device=device
        )

        self.labels = torch.full(
            (queue_size,),
            -1,
            dtype=torch.long,
            device=device
        )

        self.ptr = 0
        self.full = False

        # Diagnostic turnover counters (Reviewer 1 proof)
        self.ema_updates = 0
        self.direct_overwrites = 0

    @torch.no_grad()
    def enqueue(self, features, labels):

        features = F.normalize(
            features.detach(),
            dim=1
        )

        # Dynamic dimension check to prevent size mismatch errors
        if features.size(1) != self.features.size(1):
            self.features = torch.zeros(
                self.queue_size,
                features.size(1),
                device=features.device
            )
            self.ptr = 0
            self.full = False

        B = features.size(0)

        for i in range(B):

            idx = self.ptr
            curr_lbl = labels[i].item()

            if self.labels[idx].item() == curr_lbl:
                # Same class at slot -> EMA momentum smoothing
                self.features[idx] = (
                    self.momentum * self.features[idx]
                    +
                    (1.0 - self.momentum) * features[i]
                )
                self.features[idx] = F.normalize(self.features[idx], dim=0)
                self.ema_updates += 1
            else:
                # Class change or empty slot -> Direct overwrite
                self.features[idx] = features[i]
                self.labels[idx] = labels[i]
                self.direct_overwrites += 1

            self.ptr += 1

            if self.ptr >= self.queue_size:
                self.ptr = 0
                self.full = True

    def get(self):

        if self.full:
            return self.features, self.labels

        return (
            self.features[:self.ptr],
            self.labels[:self.ptr]
        )

    def get_distribution_stats(self, class_names=None):
        """
        Computes the complete queue distribution and turnover diagnostics:
        - Class-wise occupancy counts
        - Min entries, Avg entries per class
        - Minority class coverage %
        - EMA update frequency % vs Direct overwrite %
        """
        valid_labels = self.labels if self.full else self.labels[:self.ptr]
        valid_labels = valid_labels[valid_labels >= 0].cpu().tolist()

        counts = Counter(valid_labels)
        n_cls = self.num_classes if self.num_classes is not None else 8
        class_counts = {c: counts.get(c, 0) for c in range(n_cls)}
        active_counts = [v for v in class_counts.values() if v > 0]

        total_updates = self.ema_updates + self.direct_overwrites
        ema_freq = (self.ema_updates / total_updates * 100.0) if total_updates > 0 else 0.0
        overwrite_freq = (self.direct_overwrites / total_updates * 100.0) if total_updates > 0 else 0.0

        min_entries = min(class_counts.values()) if len(active_counts) == n_cls else 0
        avg_entries = sum(class_counts.values()) / max(1, n_cls)
        coverage_pct = (len(active_counts) / max(1, n_cls)) * 100.0

        stats = {
            "total_entries": len(valid_labels),
            "min_entries": min_entries,
            "avg_entries": avg_entries,
            "minority_coverage_pct": coverage_pct,
            "ema_update_freq_pct": ema_freq,
            "direct_overwrite_freq_pct": overwrite_freq,
            "ema_updates_count": self.ema_updates,
            "direct_overwrites_count": self.direct_overwrites,
        }

        for c in range(n_cls):
            c_name = class_names[c] if (class_names is not None and c < len(class_names)) else f"class_{c}"
            stats[f"count_{c_name}"] = class_counts[c]

        return stats


In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class TailAwareMarginKNNLoss(nn.Module):

    def __init__(
        self,
        class_weights,
        k=3,
        margin=0.2
    ):
        super().__init__()

        self.k = k
        self.margin = margin

        self.register_buffer(
            "class_weights",
            class_weights
        )

    def forward(
        self,
        features,
        labels,
        queue
    ):

        features = F.normalize(
            features,
            dim=1
        )

        q_features, q_labels = queue.get()

        if len(q_labels) < 100:
            return (
                features.sum() * 0.0,
                {}
            )

        losses = []

        pos_sim_avg = 0.0
        neg_sim_avg = 0.0
        margin_ok = 0

        for i in range(features.size(0)):

            feat = features[i]
            label = labels[i]

            sim = feat @ q_features.T

            pos_mask = (q_labels == label)
            neg_mask = (q_labels != label)

            if pos_mask.sum() < self.k:
                continue

            pos_sims = sim[pos_mask]
            neg_sims = sim[neg_mask]

            pos_topk = torch.topk(
                pos_sims,
                self.k
            ).values

            s_pos = pos_topk.mean()
            s_neg = neg_sims.max()

            loss_i = F.relu(
                self.margin
                - s_pos
                + s_neg
            )

            # NO inplace operation
            loss_i = (
                loss_i
                * self.class_weights[label]
            )

            losses.append(loss_i)

            pos_sim_avg += float(s_pos.detach())
            neg_sim_avg += float(s_neg.detach())

            if s_pos > (s_neg + self.margin):
                margin_ok += 1

        if len(losses) == 0:
            return (
                features.sum() * 0.0,
                {}
            )

        total_loss = torch.stack(
            losses
        ).mean()

        valid = len(losses)

        stats = {
            "pos_sim":
                pos_sim_avg / valid,

            "neg_sim":
                neg_sim_avg / valid,

            "margin_sat":
                100.0 * margin_ok / valid
        }

        return total_loss, stats


class CauchyPrototypeLoss(nn.Module):

    def __init__(
        self,
        num_classes,
        sigma=0.5,
        min_samples=1
    ):
        super().__init__()

        self.num_classes = num_classes
        self.sigma = sigma
        self.min_samples = min_samples

    def forward(
        self,
        features,
        labels,
        queue_features=None,
        queue_labels=None
    ):
        """
        Batch-Augmented Cauchy Prototype Dispersion Loss.
        Concatenates live mini-batch embeddings (requires_grad=True)
        with detached historical representations from the queue.
        This ensures that class prototypes are directly differentiable
        with respect to the encoder parameters theta.
        """
        if queue_features is not None and len(queue_labels) > 0:
            # Concatenate live batch features with detached queue features
            all_features = torch.cat([features, queue_features], dim=0)
            all_labels = torch.cat([labels, queue_labels], dim=0)
        else:
            all_features = features
            all_labels = labels

        prototypes = []
        valid_classes = 0

        for c in range(self.num_classes):

            mask = (all_labels == c)

            if mask.sum() < self.min_samples:
                continue

            proto = all_features[mask].mean(0)

            proto = F.normalize(
                proto,
                dim=0
            )

            prototypes.append(proto)
            valid_classes += 1

        if valid_classes < 2:
            return (
                features.sum() * 0.0,
                {
                    "proto_dist": 0.0,
                    "proto_kernel": 0.0,
                    "valid_classes": valid_classes
                }
            )

        prototypes = torch.stack(
            prototypes,
            dim=0
        )

        dist_sq = (
            (
                prototypes[:, None]
                -
                prototypes[None]
            ) ** 2
        ).sum(-1)

        eye = torch.eye(
            valid_classes,
            device=dist_sq.device,
            dtype=torch.bool
        )

        pair_dist = dist_sq[~eye]

        kernel = (
            1.0 /
            (
                1.0
                +
                pair_dist /
                self.sigma**2
            )
        )

        loss = kernel.mean()

        stats = {
            "proto_dist":
                pair_dist.mean().item(),

            "proto_kernel":
                kernel.mean().item(),

            "valid_classes":
                valid_classes
        }

        return loss, stats


In [10]:
import math

def get_proto_lambda(
    epoch,
    total_epochs=60,
    max_lambda=0.1,
    tau=15
):

    num = (
        1 -
        math.exp(-epoch / tau)
    )

    den = (
        1 -
        math.exp(
            -total_epochs / tau
        )
    )

    return max_lambda * num / den


def get_knn_lambda(
    epoch,
    total_epochs=60,
    max_lambda=0.5,
    tau=15
):

    num = (
        1 -
        math.exp(-epoch / tau)
    )

    den = (
        1 -
        math.exp(
            -total_epochs / tau
        )
    )

    return max_lambda * num / den

In [11]:
# import timm
# import torch.nn as nn
#OLD
# class ResNet18KNNTimm(nn.Module):

#     def __init__(self, num_classes):
#         super().__init__()

#         self.backbone = timm.create_model(
#             "resnet18",
#             pretrained=True,
#             num_classes=0,      # remove classifier
#             global_pool="avg"
#         )

#         feat_dim = self.backbone.num_features

#         # Disable inplace activations
#         for m in self.backbone.modules():
#             if isinstance(m, nn.ReLU):
#                 m.inplace = False

#         self.fc = nn.Linear(
#             feat_dim,
#             num_classes
#         )

#     def forward(self, x):

#         features = self.backbone(x)

#         logits = self.fc(features)

#         return features, logits

In [12]:
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F


class ResNet18KNNTimm(nn.Module):

    def __init__(self, num_classes):
        super().__init__()

        self.backbone = timm.create_model(
            "resnet18",
            pretrained=True,
            num_classes=0,
            global_pool="avg"
        )

        feat_dim = self.backbone.num_features  # 512

        # Disable inplace activations
        for m in self.backbone.modules():
            if isinstance(m, nn.ReLU):
                m.inplace = False

        # Bottleneck projection head
        self.embedding = nn.Sequential(
            nn.Linear(feat_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=False),
            nn.Dropout(0.2),
            nn.Linear(256, feat_dim)  # back to 512
        )

        # BNNeck
        self.bnneck = nn.BatchNorm1d(feat_dim)
        self.bnneck.bias.requires_grad_(False)

        # Classifier
        self.fc = nn.Linear(feat_dim, num_classes, bias=False)

    def forward(self, x):

        # Backbone feature [B,512]
        feat = self.backbone(x)

        # Geometry-aware embedding [B,512]
        emb = self.embedding(feat)

        # Normalized embedding for:
        # - KNN loss
        # - Prototype loss
        # - Retrieval metrics
        emb_norm = F.normalize(emb, p=2, dim=1)

        # Classification branch
        cls_feat = self.bnneck(emb)

        logits = self.fc(cls_feat)

        return emb_norm, logits

In [13]:
from sklearn.metrics import (
    balanced_accuracy_score,
    f1_score
)

@torch.no_grad()
def validate(
    model,
    loader,
    ce_criterion,
    device
):

    model.eval()

    running_loss = 0.0

    all_preds = []
    all_labels = []

    for images, labels in loader:

        images = images.to(device)
        labels = labels.to(device)

        _, logits = model(images)

        loss = ce_criterion(
            logits,
            labels
        )

        running_loss += loss.item()

        preds = logits.argmax(dim=1)

        all_preds.extend(
            preds.cpu().numpy()
        )

        all_labels.extend(
            labels.cpu().numpy()
        )

    val_loss = (
        running_loss / len(loader)
    )

    bal_acc = balanced_accuracy_score(
        all_labels,
        all_preds
    )

    macro_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro"
    )

    acc = (
        (torch.tensor(all_preds)
         ==
         torch.tensor(all_labels))
        .float()
        .mean()
        .item()
    )

    return {
        "loss": val_loss,
        "acc": acc * 100,
        "bal_acc": bal_acc * 100,
        "macro_f1": macro_f1 * 100
    }

In [14]:
import math
import torch
from tqdm.auto import tqdm

NUM_EPOCHS = 60

def train_one_epoch(
    model,
    loader,
    optimizer,
    scaler,
    ce_criterion,
    knn_criterion,
    proto_criterion,
    queue,
    device,
    epoch,
    total_epochs
):
    model.train()

    running_loss = 0.0
    running_ce = 0.0
    running_knn = 0.0
    running_proto = 0.0

    # Stronger Reviewer-Proof Metrics: A, B, C, D
    running_proto_grad = 0.0       # Quantity A: G_p
    running_proto_eff_grad = 0.0   # Quantity B: G_p^w
    running_rel_contrib = 0.0      # Quantity C: R_p
    running_cos_sim = 0.0          # Quantity D: cos(∇L_CE, ∇L_proto)

    total = 0
    correct = 0

    pos_sim_running = 0.0
    neg_sim_running = 0.0
    margin_running = 0.0

    knn_count = 0
    proto_step_count = 0

    lam_knn = get_knn_lambda(
        epoch,
        total_epochs=total_epochs,
        max_lambda=0.5
    )

    proto_lam = get_proto_lambda(
        epoch,
        total_epochs=total_epochs,
        max_lambda=0.5,
        tau=12
    )

    pbar = tqdm(
        loader,
        desc=f"Train Epoch [{epoch:02d}/{total_epochs:02d}]",
        dynamic_ncols=True,
        leave=True
    )

    for step, (images, labels) in enumerate(pbar, start=1):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type="cuda", dtype=torch.float16):
            features, logits = model(images)

            ce_loss = ce_criterion(logits, labels)

            knn_loss, knn_stats = knn_criterion(
                features,
                labels,
                queue
            )

            # Queue features and batch-augmented prototype loss
            q_features, q_labels = queue.get()

            proto_loss, proto_stats = proto_criterion(
                features,
                labels,
                q_features,
                q_labels
            )

            loss = (
                ce_loss
                + lam_knn * knn_loss
                + proto_lam * proto_loss
            )

        # ==============================================================
        # STRONGER REVIEWER-PROOF GRADIENT ANALYSIS (A, B, C, D)
        # ==============================================================
        proto_grad_norm = 0.0      # Quantity A: G_p = ||∇_θ L_proto||_2
        proto_eff_grad_norm = 0.0  # Quantity B: G_p^w = λ_proto * G_p
        grad_cos_sim = 0.0         # Quantity D: cos(∇L_CE, ∇L_proto)
        rel_grad_contrib = 0.0     # Quantity C: R_p = ||λ ∇L_proto|| / ||∇L_total||

        if proto_lam > 0 and proto_loss.requires_grad and proto_loss.item() > 0:
            trainable_params = [p for p in model.parameters() if p.requires_grad]

            # 1. Prototype loss gradients
            proto_grads = torch.autograd.grad(
                proto_loss,
                trainable_params,
                retain_graph=True,
                allow_unused=True
            )

            # 2. Classification loss gradients for alignment (Quantity D)
            ce_grads = torch.autograd.grad(
                ce_loss,
                trainable_params,
                retain_graph=True,
                allow_unused=True
            )

            # Quantity A, B & Alignment D computation
            dot_prod = 0.0
            p_sq_sum = 0.0
            ce_sq_sum = 0.0
            valid_p_grads = []

            for gp, gce in zip(proto_grads, ce_grads):
                if gp is not None:
                    valid_p_grads.append(gp.detach())
                    if gce is not None:
                        gce_d = gce.detach()
                        gp_d = gp.detach()
                        dot_prod += torch.sum(gp_d * gce_d).item()
                        ce_sq_sum += torch.sum(gce_d * gce_d).item()
                        p_sq_sum += torch.sum(gp_d * gp_d).item()

            if len(valid_p_grads) > 0:
                # Quantity A: G_p
                proto_grad_norm = torch.norm(
                    torch.stack([torch.norm(g, 2) for g in valid_p_grads]),
                    2
                ).item()

                # Quantity B: G_p^w
                proto_eff_grad_norm = proto_lam * proto_grad_norm

                # Quantity D: Gradient Alignment cos(∇L_CE, ∇L_proto)
                if p_sq_sum > 0 and ce_sq_sum > 0:
                    denom = math.sqrt(p_sq_sum) * math.sqrt(ce_sq_sum)
                    grad_cos_sim = dot_prod / (denom + 1e-12)

        # ==============================================================
        # Standard Combined Backward Pass & Optimization
        # ==============================================================
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)

        # Total optimization gradient norm: ||∇_θ L_total||_2
        total_grad_norm = torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            5.0
        )
        total_grad_val = float(total_grad_norm)

        # Quantity C: Relative gradient contribution R_p
        if total_grad_val > 1e-8:
            rel_grad_contrib = proto_eff_grad_norm / total_grad_val

        # Accumulate metrics when proto_loss is active
        if proto_grad_norm > 0:
            running_proto_grad += proto_grad_norm
            running_proto_eff_grad += proto_eff_grad_norm
            running_cos_sim += grad_cos_sim
            running_rel_contrib += rel_grad_contrib
            proto_step_count += 1

        scaler.step(optimizer)
        scaler.update()

        queue.enqueue(features, labels)

        preds = logits.argmax(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        running_loss += loss.item()
        running_ce += ce_loss.item()
        running_knn += knn_loss.item()
        running_proto += proto_loss.item()

        if len(knn_stats) > 0:
            pos_sim_running += knn_stats["pos_sim"]
            neg_sim_running += knn_stats["neg_sim"]
            margin_running += knn_stats["margin_sat"]
            knn_count += 1

        train_acc = 100.0 * correct / total

        # Real-time metrics displayed in progress bar
        postfix = {
            "loss": f"{running_loss/step:.3f}",
            "ce": f"{running_ce/step:.3f}",
            "proto": f"{running_proto/step:.3f}",
            "acc": f"{train_acc:.1f}%",
            "G_p": f"{proto_grad_norm:.3e}",
            "R_p": f"{rel_grad_contrib*100:.2f}%",
            "cos": f"{grad_cos_sim:+.3f}",
            "grad": f"{total_grad_val:.2f}"
        }

        if knn_count > 0:
            postfix["margin"] = f"{margin_running/knn_count:.1f}%"

        pbar.set_postfix(postfix)

    # Compute step averages
    valid_proto_steps = max(1, proto_step_count)
    avg_proto_grad = running_proto_grad / valid_proto_steps
    avg_proto_eff_grad = running_proto_eff_grad / valid_proto_steps
    avg_cos_sim = running_cos_sim / valid_proto_steps
    avg_rel_contrib = running_rel_contrib / valid_proto_steps

    epoch_stats = {
        "loss": running_loss / len(loader),
        "ce": running_ce / len(loader),
        "knn": running_knn / len(loader),
        "proto": running_proto / len(loader),
        "acc": 100.0 * correct / total,
        "knn_lambda": lam_knn,
        "proto_lambda": proto_lam,
        # Quantities A, B, C, D
        "proto_grad_norm": avg_proto_grad,          # Quantity A: G_p
        "proto_eff_grad_norm": avg_proto_eff_grad,  # Quantity B: G_p^w
        "rel_grad_contrib": avg_rel_contrib,        # Quantity C: R_p
        "grad_cos_sim": avg_cos_sim,                # Quantity D: cos(∇L_CE, ∇L_proto)
    }

    if knn_count > 0:
        epoch_stats.update({
            "pos_sim": pos_sim_running / knn_count,
            "neg_sim": neg_sim_running / knn_count,
            "margin_sat": margin_running / knn_count
        })

    return epoch_stats


In [15]:
num_classes=8

model = ResNet18KNNTimm(
    num_classes=num_classes
).to(device)


model.safetensors: reconstructing file:   0%|          |  0.00B / 46.8MB            

model.safetensors: downloading bytes:           |  0.00B            

In [16]:
# =====================================
# LOSS
# =====================================

criterion = nn.CrossEntropyLoss(
    weight=weights,
    label_smoothing=0.15
)

knn_criterion = TailAwareMarginKNNLoss(
    class_weights=weights,   # effective-number weights
    k=10,
    margin=0.2
).to(device)

proto_criterion = CauchyPrototypeLoss(
    num_classes=8,
    sigma=0.5,
    min_samples=1
)

# Feature Queue with diagnostic tracking
queue = EMAFeatureQueue(
    feature_dim=512,
    queue_size=4096,
    momentum=0.99,
    num_classes=8,
    device=device
)

# =====================================
# OPTIMIZER
# =====================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=5e-4
)


# =====================================
# SCHEDULER
# =====================================

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=60
)


# =====================================
# AMP
# =====================================

scaler = torch.amp.GradScaler("cuda")


# =====================================
# PARAMETER COUNT
# =====================================

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f"Total Params     : {total_params:,}")
print(f"Trainable Params : {trainable_params:,}")


Total Params     : 11,445,056
Trainable Params : 11,444,544


In [17]:
print(sum(p.numel() for p in model.parameters()))
print(len(list(model.parameters())))

11445056
69


In [18]:
import numpy as np
import pandas as pd

best_bal_acc = 0.0
num_classes = 8

# Track epoch-wise gradient history for Reviewer 1 diagnostic reports
gradient_history = []

# Persistent lists for per-epoch metrics and queue distributions (exported to CSV)
per_epoch_metrics = []
queue_distribution_history = []

for epoch in range(1, NUM_EPOCHS + 1):

    train_stats = train_one_epoch(
        model,
        train_loader,
        optimizer,
        scaler,
        criterion,
        knn_criterion,
        proto_criterion,
        queue,
        device,
        epoch,
        NUM_EPOCHS
    )

    val_stats = validate(
        model,
        val_loader,
        criterion,
        device
    )

    # Record complete gradient diagnostic suite (A, B, C, D)
    gradient_history.append({
        "epoch": epoch,
        "loss": train_stats["loss"],
        "proto": train_stats["proto"],
        "proto_lambda": train_stats["proto_lambda"],
        # Quantity A: Prototype Gradient Norm G_p
        "proto_grad_norm": train_stats["proto_grad_norm"],
        # Quantity B: Weighted Prototype Gradient G_p^w
        "proto_eff_grad_norm": train_stats["proto_eff_grad_norm"],
        # Quantity C: Relative Gradient Contribution R_p
        "rel_grad_contrib": train_stats["rel_grad_contrib"],
        # Quantity D: Gradient Alignment cos(∇L_CE, ∇L_proto)
        "grad_cos_sim": train_stats["grad_cos_sim"],
        "val_bal_acc": val_stats["bal_acc"],
        "val_macro_f1": val_stats["macro_f1"]
    })

    # 1. Collect full training & validation metrics row for CSV export
    epoch_metric_row = {
        "epoch": epoch,
        "train_loss": train_stats["loss"],
        "train_ce": train_stats["ce"],
        "train_knn": train_stats["knn"],
        "train_proto": train_stats["proto"],
        "train_acc": train_stats["acc"],
        "val_loss": val_stats["loss"],
        "val_acc": val_stats["acc"],
        "val_bal_acc": val_stats["bal_acc"],
        "val_macro_f1": val_stats["macro_f1"],
        "knn_lambda": train_stats["knn_lambda"],
        "proto_lambda": train_stats["proto_lambda"],
        "proto_grad_norm_G_p": train_stats["proto_grad_norm"],
        "proto_eff_grad_norm_G_p_w": train_stats["proto_eff_grad_norm"],
        "rel_grad_contrib_R_p": train_stats["rel_grad_contrib"],
        "grad_cos_sim": train_stats["grad_cos_sim"],
        "pos_sim": train_stats.get("pos_sim", 0.0),
        "neg_sim": train_stats.get("neg_sim", 0.0),
        "margin_sat": train_stats.get("margin_sat", 0.0)
    }
    per_epoch_metrics.append(epoch_metric_row)

    # 2. Collect queue training distribution row for CSV export
    q_stats = queue.get_distribution_stats(class_names=class_names)
    q_stats_row = {"epoch": epoch, **q_stats}
    queue_distribution_history.append(q_stats_row)

    # Export CSVs per-epoch incrementally (safe against unexpected container halts)
    pd.DataFrame(per_epoch_metrics).to_csv("training_metrics_per_epoch.csv", index=False)
    pd.DataFrame(queue_distribution_history).to_csv("queue_distribution_per_epoch.csv", index=False)

    print("\n" + "="*75)
    print(f"Epoch [{epoch}/{NUM_EPOCHS}]")
    print(f"Train Loss          : {train_stats['loss']:.4f}")
    print(f"Train CE            : {train_stats['ce']:.4f}")
    print(f"Train KNN           : {train_stats['knn']:.4f}")
    print(f"Train Proto (L_p)   : {train_stats['proto']:.4f}")
    print(f"Train Acc           : {train_stats['acc']:.2f}%")
    print(f"Knn Lambda          : {train_stats['knn_lambda']:.3f}")
    print(f"Proto Lambda        : {train_stats['proto_lambda']:.3f}")

    # -------------------------------------------------------------
    # STRONGER REVIEWER-PROOF GRADIENT SUITE (A, B, C, D)
    # -------------------------------------------------------------
    print(f"[A] Proto Grad Norm G_p   : {train_stats['proto_grad_norm']:.6f}  (||∇_θ L_proto||_2)")
    print(f"[B] Weighted Grad G_p^w   : {train_stats['proto_eff_grad_norm']:.6f}  (λ_proto * G_p)")
    print(f"[C] Rel Contribution R_p  : {train_stats['rel_grad_contrib']*100:.3f}%  (||λ∇L_proto|| / ||∇L_total||)")

    # Interpret cosine alignment
    cos_val = train_stats['grad_cos_sim']
    if cos_val > 0.05:
        align_note = "POSITIVE (Supports Classification)"
    elif cos_val < -0.05:
        align_note = "NEGATIVE (Gradient Conflict)"
    else:
        align_note = "NEAR-ZERO (Orthogonal Regularizer)"

    print(f"[D] Grad Alignment cos(θ) : {cos_val:+.4f}  -> {align_note}")

    # -------------------------------------------------------------
    # QUEUE DISTRIBUTION DIAGNOSTICS (REVIEWER 1 PROOF)
    # -------------------------------------------------------------
    print(f"Queue Min Entries   : {q_stats['min_entries']}  (DF tail class)")
    print(f"Queue Avg Entries   : {q_stats['avg_entries']:.1f}  (per class)")
    print(f"Minority Coverage   : {q_stats['minority_coverage_pct']:.1f}%  (Classes active)")
    print(f"EMA Update Freq     : {q_stats['ema_update_freq_pct']:.1f}%  (Smoothing)")
    print(f"Overwrite Freq      : {q_stats['direct_overwrite_freq_pct']:.1f}%  (Fresh purge)")

    if "pos_sim" in train_stats:
        print(f"Pos Sim             : {train_stats['pos_sim']:.3f}")
        print(f"Neg Sim             : {train_stats['neg_sim']:.3f}")
        print(f"Margin Sat          : {train_stats['margin_sat']:.1f}%")

    print()
    print(f"Val Loss            : {val_stats['loss']:.4f}")
    print(f"Val Acc             : {val_stats['acc']:.2f}%")
    print(f"Val BalAcc          : {val_stats['bal_acc']:.2f}%")
    print(f"Val MacroF1         : {val_stats['macro_f1']:.2f}%")

    if val_stats["bal_acc"] > best_bal_acc:
        best_bal_acc = val_stats["bal_acc"]
        torch.save(
            {
                "epoch": epoch,
                "model": model.state_dict(),
                "best_bal_acc": best_bal_acc
            },
            "best_model.pth"
        )
        print(f"✓ Best Model Saved (BalAcc={best_bal_acc:.2f})")

    print("="*75)

    # =============================================================
    # PER 10-EPOCH STRONGER REVIEWER-PROOF REPORT (EVERY 10 EPOCHS)
    # =============================================================
    if epoch % 10 == 0:
        window_10 = gradient_history[-10:]
        q_window_10 = queue_distribution_history[-10:]
        avg_proto_loss = np.mean([e["proto"] for e in window_10])
        avg_proto_lam = np.mean([e["proto_lambda"] for e in window_10])
        avg_gp = np.mean([e["proto_grad_norm"] for e in window_10])
        avg_gp_w = np.mean([e["proto_eff_grad_norm"] for e in window_10])
        avg_rp = np.mean([e["rel_grad_contrib"] for e in window_10])
        avg_cos = np.mean([e["grad_cos_sim"] for e in window_10])
        avg_q_min = np.mean([q["min_entries"] for q in q_window_10])
        avg_q_avg = np.mean([q["avg_entries"] for q in q_window_10])
        avg_ema_freq = np.mean([q["ema_update_freq_pct"] for q in q_window_10])

        print("\n" + "#"*92)
        print(f"  [REVIEWER 1 PROOF] 10-EPOCH GRADIENT SUITE & QUEUE REPORT (EPOCHS {epoch-9} to {epoch})")
        print("#"*92)
        print(f"{'Epoch':<6} | {'L_p':<7} | {'λ_p':<7} | {'G_p (||∇L_p||)':<16} | {'G_p^w (λ||∇||)':<15} | {'R_p (Rel %)':<12} | {'cos(∇L_CE,∇L_p)':<16}")
        print("-" * 92)
        for e in window_10:
            print(f"{e['epoch']:<6} | {e['proto']:<7.4f} | {e['proto_lambda']:<7.4f} | {e['proto_grad_norm']:<16.6f} | {e['proto_eff_grad_norm']:<15.6f} | {e['rel_grad_contrib']*100:<11.3f}% | {e['grad_cos_sim']:<+16.4f}")
        print("-" * 92)
        print(f"10-Epoch Window Summary (Epochs {epoch-9} - {epoch}):")
        print(f"  • [A] Mean Prototype Gradient Norm (G_p)    : {avg_gp:.6f}  (Non-zero encoder updates)")
        print(f"  • [B] Mean Weighted Gradient (G_p^w)        : {avg_gp_w:.6f}  (Warmup stabilization)")
        print(f"  • [C] Mean Relative Contribution (R_p)      : {avg_rp*100:.3f}%  (Share of total optimization)")
        print(f"  • [D] Mean Gradient Alignment (cos θ)       : {avg_cos:+.4f}  ({'Alignment supports CE' if avg_cos > 0 else 'Independent regularizer'})")
        print(f"  • Queue Dynamics: Mean Min Entries = {avg_q_min:.1f} | Mean Avg Entries = {avg_q_avg:.1f} | Mean EMA Freq = {avg_ema_freq:.1f}%")
        print("#"*92 + "\n")

# =============================================================
# PER 10-EPOCH CSV AGGREGATION & EXPORT (AFTER TRAINING FINISHES)
# =============================================================
per_10_epoch_summary = []
for i in range(0, len(per_epoch_metrics), 10):
    chunk = per_epoch_metrics[i:i+10]
    q_chunk = queue_distribution_history[i:i+10]
    win_label = f"Epochs {chunk[0]['epoch']:02d}-{chunk[-1]['epoch']:02d}"
    summary_row = {
        "milestone_window": win_label,
        "start_epoch": chunk[0]["epoch"],
        "end_epoch": chunk[-1]["epoch"],
        "mean_train_loss": float(np.mean([e["train_loss"] for e in chunk])),
        "mean_train_acc": float(np.mean([e["train_acc"] for e in chunk])),
        "mean_val_loss": float(np.mean([e["val_loss"] for e in chunk])),
        "mean_val_acc": float(np.mean([e["val_acc"] for e in chunk])),
        "mean_val_bal_acc": float(np.mean([e["val_bal_acc"] for e in chunk])),
        "best_val_bal_acc": float(np.max([e["val_bal_acc"] for e in chunk])),
        "mean_val_macro_f1": float(np.mean([e["val_macro_f1"] for e in chunk])),
        "best_val_macro_f1": float(np.max([e["val_macro_f1"] for e in chunk])),
        "mean_proto_loss_L_p": float(np.mean([e["train_proto"] for e in chunk])),
        "mean_proto_lambda": float(np.mean([e["proto_lambda"] for e in chunk])),
        "mean_G_p": float(np.mean([e["proto_grad_norm_G_p"] for e in chunk])),
        "mean_G_p_w": float(np.mean([e["proto_eff_grad_norm_G_p_w"] for e in chunk])),
        "mean_R_p_pct": float(np.mean([e["rel_grad_contrib_R_p"] for e in chunk]) * 100),
        "mean_cos_alignment": float(np.mean([e["grad_cos_sim"] for e in chunk])),
        "mean_queue_min_entries": float(np.mean([q["min_entries"] for q in q_chunk])),
        "mean_queue_avg_entries": float(np.mean([q["avg_entries"] for q in q_chunk])),
        "mean_queue_ema_freq_pct": float(np.mean([q["ema_update_freq_pct"] for q in q_chunk])),
    }
    per_10_epoch_summary.append(summary_row)

df_10_epochs = pd.DataFrame(per_10_epoch_summary)
df_10_epochs.to_csv("metrics_per_10_epochs.csv", index=False)

print("\n" + "="*96)
print("  ALL METRICS & QUEUE DISTRIBUTIONS SUCCESSFULLY EXPORTED TO CSV:")
print("  1. 'training_metrics_per_epoch.csv'   -> Full per-epoch training & validation metrics")
print("  2. 'metrics_per_10_epochs.csv'        -> Aggregated 10-epoch milestone summaries")
print("  3. 'queue_distribution_per_epoch.csv' -> Full per-epoch queue class distribution & turnover dynamics")
print("="*96 + "\n")


Train Epoch [01/60]:   0%|          | 0/158 [00:18<?, ?it/s]


Epoch [1/60]
Train Loss          : 2.0626
Train CE            : 2.0393
Train KNN           : 0.1221
Train Proto (L_p)   : 0.4781
Train Acc           : 57.84%
Knn Lambda          : 0.033
Proto Lambda        : 0.040
[A] Proto Grad Norm G_p   : 0.150497  (||∇_θ L_proto||_2)
[B] Weighted Grad G_p^w   : 0.006057  (λ_proto * G_p)
[C] Rel Contribution R_p  : 0.175%  (||λ∇L_proto|| / ||∇L_total||)
[D] Grad Alignment cos(θ) : +0.1716  -> POSITIVE (Supports Classification)
Queue Min Entries   : 29  (DF tail class)
Queue Avg Entries   : 512.0  (per class)
Minority Coverage   : 100.0%  (Classes active)
EMA Update Freq     : 25.8%  (Smoothing)
Overwrite Freq      : 74.2%  (Fresh purge)
Pos Sim             : 0.529
Neg Sim             : 0.575
Margin Sat          : 0.0%

Val Loss            : 1.8723
Val Acc             : 66.92%
Val BalAcc          : 46.67%
Val MacroF1         : 42.87%
✓ Best Model Saved (BalAcc=46.67)


Train Epoch [02/60]:   0%|          | 0/158 [00:00<?, ?it/s]


Epoch [2/60]
Train Loss          : 1.8982
Train CE            : 1.8627
Train KNN           : 0.1158
Train Proto (L_p)   : 0.3651
Train Acc           : 67.37%
Knn Lambda          : 0.064
Proto Lambda        : 0.077
[A] Proto Grad Norm G_p   : 0.065921  (||∇_θ L_proto||_2)
[B] Weighted Grad G_p^w   : 0.005094  (λ_proto * G_p)
[C] Rel Contribution R_p  : 0.189%  (||λ∇L_proto|| / ||∇L_total||)
[D] Grad Alignment cos(θ) : +0.2020  -> POSITIVE (Supports Classification)
Queue Min Entries   : 34  (DF tail class)
Queue Avg Entries   : 512.0  (per class)
Minority Coverage   : 100.0%  (Classes active)
EMA Update Freq     : 28.7%  (Smoothing)
Overwrite Freq      : 71.3%  (Fresh purge)
Pos Sim             : 0.582
Neg Sim             : 0.615
Margin Sat          : 0.0%

Val Loss            : 1.8248
Val Acc             : 69.29%
Val BalAcc          : 52.30%
Val MacroF1         : 49.33%
✓ Best Model Saved (BalAcc=52.30)


Train Epoch [03/60]:   0%|          | 0/158 [00:00<?, ?it/s]

...


Epoch [58/60]
Train Loss          : 1.3092
Train CE            : 1.2424
Train KNN           : 0.0193
Train Proto (L_p)   : 0.1144
Train Acc           : 99.11%
Knn Lambda          : 0.499
Proto Lambda        : 0.499
[A] Proto Grad Norm G_p   : 0.000974  (||∇_θ L_proto||_2)
[B] Weighted Grad G_p^w   : 0.000487  (λ_proto * G_p)
[C] Rel Contribution R_p  : 0.091%  (||λ∇L_proto|| / ||∇L_total||)
[D] Grad Alignment cos(θ) : +0.0165  -> NEAR-ZERO (Orthogonal Regularizer)
Queue Min Entries   : 37  (DF tail class)
Queue Avg Entries   : 512.0  (per class)
Minority Coverage   : 100.0%  (Classes active)
EMA Update Freq     : 32.0%  (Smoothing)
Overwrite Freq      : 68.0%  (Fresh purge)
Pos Sim             : 0.922
Neg Sim             : 0.731
Margin Sat          : 45.3%

Val Loss            : 1.6394
Val Acc             : 80.46%
Val BalAcc          : 69.82%
Val MacroF1         : 72.58%


Train Epoch [59/60]:   0%|          | 0/158 [00:00<?, ?it/s]


Epoch [59/60]
Train Loss          : 1.3069
Train CE            : 1.2412
Train KNN           : 0.0170
Train Proto (L_p)   : 0.1146
Train Acc           : 99.17%
Knn Lambda          : 0.499
Proto Lambda        : 0.500
[A] Proto Grad Norm G_p   : 0.000937  (||∇_θ L_proto||_2)
[B] Weighted Grad G_p^w   : 0.000468  (λ_proto * G_p)
[C] Rel Contribution R_p  : 0.088%  (||λ∇L_proto|| / ||∇L_total||)
[D] Grad Alignment cos(θ) : +0.0147  -> NEAR-ZERO (Orthogonal Regularizer)
Queue Min Entries   : 40  (DF tail class)
Queue Avg Entries   : 512.0  (per class)
Minority Coverage   : 100.0%  (Classes active)
EMA Update Freq     : 32.0%  (Smoothing)
Overwrite Freq      : 68.0%  (Fresh purge)
Pos Sim             : 0.921
Neg Sim             : 0.718
Margin Sat          : 50.9%

Val Loss            : 1.6078
Val Acc             : 81.68%
Val BalAcc          : 73.00%
Val MacroF1         : 75.87%


Train Epoch [60/60]:   0%|          | 0/158 [00:00<?, ?it/s]


Epoch [60/60]
Train Loss          : 1.3022
Train CE            : 1.2377
Train KNN           : 0.0141
Train Proto (L_p)   : 0.1148
Train Acc           : 99.37%
Knn Lambda          : 0.500
Proto Lambda        : 0.500
[A] Proto Grad Norm G_p   : 0.000958  (||∇_θ L_proto||_2)
[B] Weighted Grad G_p^w   : 0.000479  (λ_proto * G_p)
[C] Rel Contribution R_p  : 0.093%  (||λ∇L_proto|| / ||∇L_total||)
[D] Grad Alignment cos(θ) : +0.0134  -> NEAR-ZERO (Orthogonal Regularizer)
Queue Min Entries   : 37  (DF tail class)
Queue Avg Entries   : 512.0  (per class)
Minority Coverage   : 100.0%  (Classes active)
EMA Update Freq     : 32.0%  (Smoothing)
Overwrite Freq      : 68.0%  (Fresh purge)
Pos Sim             : 0.924
Neg Sim             : 0.709
Margin Sat          : 55.6%

Val Loss            : 1.5803
Val Acc             : 83.22%
Val BalAcc          : 74.40%
Val MacroF1         : 76.32%
✓ Best Model Saved (BalAcc=74.40)

################################################################################

In [19]:
# ==============================================================================
# INSPECT & PREVIEW EXPORTED CSV METRICS & QUEUE DISTRIBUTIONS
# ==============================================================================
import pandas as pd

print('='*80)
print('1. PREVIEW: training_metrics_per_epoch.csv')
print('='*80)
try:
    df_epochs = pd.read_csv('training_metrics_per_epoch.csv')
    display(df_epochs.head(10))
except Exception as e:
    print('Error loading training_metrics_per_epoch.csv:', e)

print('
' + '='*80)
print('2. PREVIEW: metrics_per_10_epochs.csv (Milestone Summary for Rebuttal)')
print('='*80)
try:
    df_10 = pd.read_csv('metrics_per_10_epochs.csv')
    display(df_10)
except Exception as e:
    print('Error loading metrics_per_10_epochs.csv:', e)

print('
' + '='*80)
print('3. PREVIEW: queue_distribution_per_epoch.csv (Class Distribution & Dynamics)')
print('='*80)
try:
    df_queue = pd.read_csv('queue_distribution_per_epoch.csv')
    display(df_queue.head(10))
except Exception as e:
    print('Error loading queue_distribution_per_epoch.csv:', e)


SyntaxError: unterminated string literal (detected at line 15) (415752009.py, line 15)

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

from sklearn.manifold import TSNE
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
)
from sklearn.neighbors import NearestNeighbors
from scipy.spatial.distance import cdist


@torch.no_grad()
def extract_features(model, loader, device):

    model.eval()

    feats = []
    labels = []
    preds = []

    for images, targets in loader:

        images = images.to(device)

        features, logits = model(images)

        features = torch.nn.functional.normalize(
            features,
            p=2,
            dim=1
        )

        pred = logits.argmax(dim=1)

        feats.append(features.cpu())
        labels.append(targets.cpu())
        preds.append(pred.cpu())

    feats = torch.cat(feats).numpy()
    labels = torch.cat(labels).numpy()
    preds = torch.cat(preds).numpy()

    print(f"Feature shape: {feats.shape}")

    return feats, labels, preds


def knn_purity(features, labels, k=10):
    nbrs = NearestNeighbors(
        n_neighbors=k + 1,
        metric="cosine"
    ).fit(features)

    _, idx = nbrs.kneighbors(features)

    idx = idx[:, 1:]

    purity = []

    for i in range(len(features)):
        neigh_labels = labels[idx[i]]
        purity.append(
            np.mean(neigh_labels == labels[i])
        )

    return np.mean(purity)


def class_separation_stats(features, labels):

    classes = np.unique(labels)

    centroids = []
    intra_cos = []

    for c in classes:

        cls_feat = features[labels == c]

        centroid = cls_feat.mean(axis=0)
        centroid /= np.linalg.norm(centroid) + 1e-8

        centroids.append(centroid)

        sims = cls_feat @ centroid
        intra_cos.append(np.mean(sims))

    centroids = np.stack(centroids)

    inter_cos = centroids @ centroids.T

    mask = ~np.eye(len(classes), dtype=bool)
    inter_vals = inter_cos[mask]

    return {
        "mean_intra_cos": np.mean(intra_cos),
        "std_intra_cos": np.std(intra_cos),
        "mean_inter_cos": np.mean(inter_vals),
        "min_inter_cos": np.min(inter_vals),
        "max_inter_cos": np.max(inter_vals),
        "margin": np.mean(intra_cos) - np.mean(inter_vals)
    }

def prototype_alignment(features, labels):

    classes = np.unique(labels)

    protos = []

    for c in classes:
        proto = features[labels == c].mean(axis=0)
        proto /= np.linalg.norm(proto) + 1e-8
        protos.append(proto)

    protos = np.stack(protos)

    alignments = []

    for c in classes:

        cls_feat = features[labels == c]

        sim = cls_feat @ protos[c]

        alignments.extend(sim)

    return np.mean(alignments)


def evaluate_embedding(model, loader, device):

    features, labels, preds = extract_features(
        model,
        loader,
        device
    )

    print("=" * 60)
    print("Embedding Statistics")
    print("=" * 60)

    sil = silhouette_score(
        features,
        labels
    )

    dbi = davies_bouldin_score(
        features,
        labels
    )

    chi = calinski_harabasz_score(
        features,
        labels
    )

    purity = knn_purity(
        features,
        labels,
        k=10
    )

    sep_stats = class_separation_stats(
        features,
        labels
    )

    proto_align = prototype_alignment(
        features,
        labels
    )

    print(f"Silhouette Score      : {sil:.4f}")
    print(f"Davies-Bouldin Index  : {dbi:.4f}")
    print(f"Calinski-Harabasz     : {chi:.2f}")
    print(f"KNN Purity (k=10)     : {purity:.4f}")
    print(f"Prototype Alignment   : {proto_align:.4f}")

    print("\nClass Separation")

    for k, v in sep_stats.items():
        print(f"{k:20s}: {v:.4f}")

    return features, labels


from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import numpy as np


def plot_tsne(features, labels):

    pca = PCA(n_components=50)
    feats_pca = pca.fit_transform(features)

    tsne = TSNE(
        n_components=2,
        perplexity=30,
        learning_rate="auto",
        init="pca",
        random_state=42,
        max_iter=3000
    )

    emb_2d = tsne.fit_transform(feats_pca)

    plt.figure(figsize=(12, 10))

    classes = np.unique(labels)

    for c in classes:

        idx = labels == c

        plt.scatter(
            emb_2d[idx, 0],
            emb_2d[idx, 1],
            s=15,
            alpha=0.7,
            label=f"Class {c}"
        )

    plt.legend()
    plt.title("t-SNE of Test Embeddings")
    plt.tight_layout()
    plt.show()

    return emb_2d


def classwise_report(features, labels):

    classes = np.unique(labels)

    print("\nClasswise Compactness")
    print("-" * 50)

    for c in classes:

        cls_feat = features[labels == c]

        center = cls_feat.mean(axis=0)

        dist = np.linalg.norm(
            cls_feat - center,
            axis=1
        )

        print(
            f"Class {c}: "
            f"mean={dist.mean():.4f}, "
            f"std={dist.std():.4f}, "
            f"max={dist.max():.4f}"
        )




In [ ]:
import torch
import numpy as np
from sklearn.metrics import recall_score, balanced_accuracy_score, accuracy_score, f1_score

@torch.no_grad()
def evaluate_loader(model, data_loader, device):
    """
    Computes:
    1. Overall (Normal) Accuracy (%)
    2. Per-Class Recalls (%)
    3. Macro Recall / Macro F1 (%)
    4. Balanced Accuracy (%)
    """
    model.eval()
    all_preds = []
    all_targets = []

    for batch in data_loader:
        if isinstance(batch, (list, tuple)):
            images, labels = batch[0], batch[1]
        elif isinstance(batch, dict):
            images, labels = batch['image'], batch['label']

        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        if isinstance(outputs, tuple):
            logits = outputs[0] if outputs[0].shape[-1] < outputs[1].shape[-1] else outputs[1]
        elif isinstance(outputs, dict):
            logits = outputs['logits'] if 'logits' in outputs else list(outputs.values())[0]
        else:
            logits = outputs

        if labels.ndim > 1:
            labels = torch.argmax(labels, dim=1)

        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(labels.cpu().numpy())

    all_preds = np.array(all_preds)
    all_targets = np.array(all_targets)

    # 1. Overall / Normal Accuracy (Micro Accuracy)
    overall_acc = accuracy_score(all_targets, all_preds) * 100.0

    # 2. Per-Class Recalls
    per_class_recall = recall_score(all_targets, all_preds, average=None, zero_division=0) * 100.0

    # 3. Macro Recall (Mean of Per-Class Recalls)
    macro_recall = np.mean(per_class_recall)

    # 4. Balanced Accuracy (scikit-learn)
    balanced_acc = balanced_accuracy_score(all_targets, all_preds) * 100.0

    # 5. Macro F1 Score
    macro_f1 = f1_score(all_targets, all_preds, average='macro', zero_division=0) * 100.0

    return {
        'overall_acc': overall_acc,
        'per_class_recall': per_class_recall,
        'macro_recall': macro_recall,
        'balanced_acc': balanced_acc,
        'macro_f1': macro_f1
    }

def print_results(title, metrics, class_names):
    print(f"\n{'='*45}")
    print(f" {title.upper()} METRICS")
    print(f"{'='*45}")
    print(f"{'Class Name':<12} | {'Recall (%)':<10}")
    print(f"{'-'*30}")
    for name, rec in zip(class_names, metrics['per_class_recall']):
        print(f"{name:<12} | {rec:6.2f}%")
    print(f"{'-'*30}")
    print(f"{'Overall (Normal) Acc':<22} : {metrics['overall_acc']:6.2f}%")
    print(f"{'Macro Recall (Mean Rec)':<22} : {metrics['macro_recall']:6.2f}%")
    print(f"{'Balanced Accuracy':<22} : {metrics['balanced_acc']:6.2f}%")
    print(f"{'Macro F1 Score':<22} : {metrics['macro_f1']:6.2f}%")
    print(f"{'='*45}\n")


# ========================================================
# EXECUTION
# ========================================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# # Class names for ISIC 2018 (7 classes)
# class_names = ['MEL', 'NV', 'BCC', 'AKIEC', 'BKL', 'DF', 'VASC']

# For ISIC 2019 (8 classes):
class_names = ['MEL', 'NV', 'BCC', 'AK', 'BKL', 'DF', 'VASC', 'SCC']

val_results = evaluate_loader(model, val_loader, device)
print_results("Validation Set", val_results, class_names)

test_results = evaluate_loader(model, test_loader, device)
print_results("Test Set", test_results, class_names)

In [ ]:
features, labels = evaluate_embedding(
    model,
    val_loader,
    device
)

plot_tsne(
    features,
    labels
)

classwise_report(
    features,
    labels
)

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import numpy as np


def plot_tsne_compact(features, labels, class_names=None):

    n_pca = min(
        50,
        features.shape[1],
        features.shape[0] - 1
    )

    feats_pca = PCA(
        n_components=n_pca,
        random_state=42
    ).fit_transform(features)

    tsne = TSNE(
        n_components=2,
        perplexity=min(30, len(features)//10),
        learning_rate="auto",
        init="pca",
        max_iter=3000,
        random_state=42
    )

    emb_2d = tsne.fit_transform(feats_pca)

    plt.figure(figsize=(8, 8), dpi=300)

    classes = np.unique(labels)

    cmap = plt.cm.tab10

    for i, c in enumerate(classes):

        idx = labels == c

        plt.scatter(
            emb_2d[idx, 0],
            emb_2d[idx, 1],
            s=8,
            alpha=0.75,
            color=cmap(i),
            edgecolors="none"
        )

        cx = emb_2d[idx, 0].mean()
        cy = emb_2d[idx, 1].mean()

        label = (
            class_names[c]
            if class_names is not None
            else str(c)
        )

        plt.text(
            cx,
            cy,
            label,
            fontsize=10,
            weight="bold"
        )

    plt.xticks([])
    plt.yticks([])

    plt.box(False)

    plt.title(
        "t-SNE of Test Embeddings",
        fontsize=14,
        pad=10
    )

    plt.tight_layout()
    plt.show()

    return emb_2d

In [ ]:
plot_tsne(
    features,
    labels
)
